In [ ]:
import pandas as pd
import numpy as np

# new things: merging and its different methods, np.select, np.maximum, vectorisation, ~ for not, sort_values

In [ ]:
# Seed for reproducibility
np.random.seed(42)

# Table 1: Customers
customers_data = {
    'CustomerID': [1, 2, 3, 4, 5],
    'Name': ['Alice', 'Bob', 'Charlie', 'Dana', 'Eli'],
    'Tier': ['Gold', 'Silver', 'Bronze', 'Gold', 'Silver'],
    'Tier_Discount': [0.20, 0.10, 0.05, 0.20, 0.10]
}
df_customers = pd.DataFrame(customers_data)

# Table 2: Orders
# NumPy integration: generate order amounts rounded to 2 decimal places
raw_amounts = np.round(np.random.uniform(30.0, 350.0, size=7), 2)

orders_data = {
    'OrderID': [1001, 1002, 1003, 1004, 1005, 1006, 1007],
    'CustomerID': [1, 2, 1, 3, 6, 2, 4],  # Note: CustomerID 6 does not exist in df_customers
    'Order_Amount': raw_amounts,
    'Coupon_Code': ['SAVE10', None, 'SAVE20', None, 'SAVE10', None, 'SAVE10']
}
df_orders = pd.DataFrame(orders_data)

In [ ]:
# pd.merge() df_orders and df_customers to do inner merge
# pd.merge( , how="outer") to include everything with appropriate NaN values where missing data. how = 'inner' is the same as above
# pd.merge( , how='left') to keep only from left dataframe, how='right' for right dataframe
# pd.merge( , on='customerid') to merge on customer id

# Can also use dataframe method --> df_orders.join(df_customers). all conditions as with pd.merge()

print(pd.merge(df_orders, df_customers, on='CustomerID', how='inner'))
df_merged = pd.merge(df_orders, df_customers, on='CustomerID', how='left')
df_merged["Is_Guest"] = ~df_merged['CustomerID'].isin(df_customers['CustomerID'])
df_merged = df_merged.fillna({"Tier_Discount": 0.0})
condlist = [
    df_merged["Coupon_Code"] == "SAVE10",
    df_merged["Coupon_Code"] == "SAVE20"
    ]
choicelist = (0.1, 0.2)
df_merged.insert(4, "Coupon_Discount", np.select(condlist, choicelist, 0))


In [60]:
# def order_cost(coup_discount, tier_discount, order_amount):
#     final_discount_rate = np.maximum(coup_discount, tier_discount)
#     final_amount = order_amount * (1-final_discount_rate)
#     return final_amount

# vectorized_order_cost = np.vectorize(order_cost)
df_merged["Final_Discount_Rate"] = np.maximum(df_merged["Coupon_Discount"], df_merged["Tier_Discount"])
df_merged["Final_Amount"] = df_merged["Order_Amount"] * (1-df_merged["Final_Discount_Rate"])
percentile_thresh = np.percentile(df_merged["Final_Amount"], 75)
df_merged[df_merged["Final_Amount"] >= percentile_thresh]

,OrderID,CustomerID,Order_Amount,Coupon_Code,Coupon_Discount,Name,Tier,Tier_Discount,Is_Guest,Final_Amount,Final_Discount_Rate
1,1002,2,334.23,None,0.0,Bob,Silver,0.1,False,300.807,0.1
2,1003,1,264.24,SAVE20,0.2,Alice,Gold,0.2,False,211.392,0.2


In [58]:
df_tier_summary = df_merged[~df_merged["Is_Guest"]].groupby(["Tier"]).agg({
    "Final_Amount": ["sum", "mean"],
    "OrderID": "nunique"
    })
df_tier_summary = df_tier_summary.sort_values(by=("Final_Amount", "sum"), ascending=False)
df_tier_summary

Final_Amount             OrderID
                sum        mean nunique
Tier                                   
Silver     372.7350  186.367500       2
Gold       370.1440  123.381333       3
Bronze     210.4915  210.491500       1